<a href="https://colab.research.google.com/github/Anto-sujin/spam_mail_detection/blob/main/spam_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd

In [3]:
df=pd.read_csv("/content/SMSSpamCollection",sep="\t",header=None,names=["label", "message"])

In [4]:
df.head()

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [5]:
df.shape

(5572, 2)

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   label    5572 non-null   object
 1   message  5572 non-null   object
dtypes: object(2)
memory usage: 87.2+ KB


In [7]:
df['label'].value_counts()

,count
label,
ham,4825
spam,747


In [8]:
mapping ={"ham":0,"spam":1}

In [9]:
df['label']=df['label'].map(mapping)

In [10]:
df['label'].value_counts()

,count
label,
0,4825
1,747


In [11]:
df.isnull().sum()

,0
label,0
message,0


In [12]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   label    5572 non-null   int64 
 1   message  5572 non-null   object
dtypes: int64(1), object(1)
memory usage: 87.2+ KB


In [13]:
df['message']=df['message'].str.lower().str.strip()

In [14]:
df.head()

,label,message
0,0,"go until jurong point, crazy.. available only ..."
1,0,ok lar... joking wif u oni...
2,1,free entry in 2 a wkly comp to win fa cup fina...
3,0,u dun say so early hor... u c already then say...
4,0,"nah i don't think he goes to usf, he lives aro..."


In [15]:
import string

In [16]:
string.punctuation

'!"#$%&\'()*+,-./:;<=>?@[\\]^_`{|}~'

In [17]:
df['message']=df['message'].replace(r"[^\w\s]","",regex=True)

In [18]:
from sklearn.model_selection import train_test_split

In [19]:
X=df.iloc[:,-1]
y=df.iloc[:,:-1]

In [20]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

In [21]:
print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 4457
Testing samples: 1115


In [22]:
a = len(y_train)
b = y_train.value_counts()[0]
c = y_train.value_counts()[1]

In [23]:
print(f'NOT SPAM PERCENTAGE:{int((b/a)*100)}')

NOT SPAM PERCENTAGE:86


In [24]:
print(f'SPAM PERCENTAGE :{(int((c/a)*100))}')

SPAM PERCENTAGE :13


In [25]:
a = len(y_test)
b = y_test.value_counts()[0]
c = y_test.value_counts()[1]


In [26]:
print(int((b/a)*100))

86


In [27]:
print(int((c/a)*100))

13


In [28]:
y_test.value_counts()

,count
label,
0,966
1,149


In [29]:
from sklearn.feature_extraction.text import TfidfVectorizer
vec = TfidfVectorizer()
X_trainD=vec.fit_transform(X_train)
X_testD = vec.transform(X_test)

In [49]:
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier


In [31]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_auc_score,
    average_precision_score
)

In [32]:
metrics = [
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_auc_score,
    average_precision_score
]

In [33]:
models = {
    "Logistic Regression": LogisticRegression(),
    "Naive Bayes": MultinomialNB(),
    "Support Vector Machine": SVC(),
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest": RandomForestClassifier(),
    "K-Nearest Neighbors": KNeighborsClassifier(),
    "Gradient Boosting": GradientBoostingClassifier(),
    "XGBoost": XGBClassifier(),
    "LightGBM": LGBMClassifier(),

}

In [35]:
y_train = y_train.values.ravel()
y_test = y_test.values.ravel()

In [36]:
 results = []

for model_name, model in models.items():

    model.fit(X_trainD, y_train)

    pre = model.predict(X_testD)

    if hasattr(model, "predict_proba"):
        score = model.predict_proba(X_testD)[:, 1]
    else:
        score = model.decision_function(X_testD)

    results.append({
        "Model": model_name,
        "Accuracy": accuracy_score(y_test, pre),
        "Precision": precision_score(y_test, pre),
        "Recall": recall_score(y_test, pre),
        "F1-Score": f1_score(y_test, pre),
        "ROC-AUC": roc_auc_score(y_test, score),
        "PR-AUC": average_precision_score(y_test, score)
    })

results_df = pd.DataFrame(results)



[LightGBM] [Info] Number of positive: 598, number of negative: 3859
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.011640 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 12795
[LightGBM] [Info] Number of data points in the train set: 4457, number of used features: 453
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.134171 -> initscore=-1.864573
[LightGBM] [Info] Start training from score -1.864573


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [37]:
print(results_df)

                    Model  Accuracy  Precision    Recall  F1-Score   ROC-AUC  \
0     Logistic Regression  0.968610   0.991379  0.771812  0.867925  0.987578   
1             Naive Bayes  0.956951   1.000000  0.677852  0.808000  0.974961   
2  Support Vector Machine  0.985650   1.000000  0.892617  0.943262  0.990913   
3           Decision Tree  0.956054   0.884615  0.771812  0.824373  0.878142   
4           Random Forest  0.974888   1.000000  0.812081  0.896296  0.996026   
5     K-Nearest Neighbors  0.921076   1.000000  0.409396  0.580952  0.961878   
6       Gradient Boosting  0.973094   0.983740  0.812081  0.889706  0.979011   
7                 XGBoost  0.974888   0.961832  0.845638  0.900000  0.978851   
8                LightGBM  0.982063   0.984962  0.879195  0.929078  0.982485   

     PR-AUC  
0  0.968730  
1  0.947342  
2  0.976664  
3  0.713250  
4  0.982630  
5  0.926376  
6  0.949500  
7  0.953442  
8  0.962698  


In [39]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV


In [40]:
# ==========================================
# 1. SVM Pipeline
# ==========================================

svm_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("model", SVC())
])

svm_param_grid = {
    "tfidf__ngram_range": [(1, 1), (1, 2)],
    "tfidf__min_df": [1, 2, 3],
    "tfidf__max_df": [0.95, 1.0],
    "tfidf__sublinear_tf": [False, True],

    "model__C": [0.1, 1, 10],
    "model__kernel": ["linear", "rbf"]
}

In [41]:
svm_grid = GridSearchCV(
    svm_pipeline,
    svm_param_grid,
    cv=5,
    scoring="f1",
    n_jobs=-1,
    verbose=1
)

In [42]:
# ==========================================
# 2. Random Forest Pipeline
# ==========================================

rf_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("model", RandomForestClassifier(random_state=42))
])

rf_param_grid = {
    "tfidf__ngram_range": [(1, 1), (1, 2)],
    "tfidf__min_df": [1, 2, 3],
    "tfidf__max_df": [0.95, 1.0],
    "tfidf__sublinear_tf": [False, True],

    "model__n_estimators": [100, 200],
    "model__max_depth": [None, 20],
    "model__min_samples_split": [2, 5]
}
rf_grid = GridSearchCV(
    rf_pipeline,
    rf_param_grid,
    cv=5,
    scoring="f1",
    n_jobs=-1,
    verbose=1
)

In [43]:
# ==========================================
# 3. LightGBM Pipeline
# ==========================================

lgbm_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("model", LGBMClassifier(
        random_state=42,
        verbosity=-1
    ))
])

lgbm_param_grid = {
    "tfidf__ngram_range": [(1, 1), (1, 2)],
    "tfidf__min_df": [1, 2, 3],
    "tfidf__max_df": [0.95, 1.0],
    "tfidf__sublinear_tf": [False, True],

    "model__n_estimators": [100, 200],
    "model__learning_rate": [0.05, 0.1],
    "model__num_leaves": [31, 50]
}


# ==========================================
# 4. GridSearchCV
# ==========================================





lgbm_grid = GridSearchCV(
    lgbm_pipeline,
    lgbm_param_grid,
    cv=5,
    scoring="f1",
    n_jobs=-1,
    verbose=1
)


In [46]:
svm_grid.fit(X_train, y_train)
rf_grid.fit(X_train, y_train)
lgbm_grid.fit(X_train, y_train)

Fitting 5 folds for each of 144 candidates, totalling 720 fits
Fitting 5 folds for each of 192 candidates, totalling 960 fits
Fitting 5 folds for each of 192 candidates, totalling 960 fits


/usr/local/lib/python3.13/dist-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('tfidf', TfidfVectorizer()),
                                       ('model',
                                        LGBMClassifier(random_state=42,
                                                       verbosity=-1))]),
             n_jobs=-1,
             param_grid={'model__learning_rate': [0.05, 0.1],
                         'model__n_estimators': [100, 200],
                         'model__num_leaves': [31, 50],
                         'tfidf__max_df': [0.95, 1.0],
                         'tfidf__min_df': [1, 2, 3],
                         'tfidf__ngram_range': [(1, 1), (1, 2)],
                         'tfidf__sublinear_tf': [False, True]},
             scoring='f1', verbose=1)

In [47]:
print("\n===== SVM =====")
print("Best Parameters:", svm_grid.best_params_)
print("Best CV F1:", svm_grid.best_score_)

print("\n===== Random Forest =====")
print("Best Parameters:", rf_grid.best_params_)
print("Best CV F1:", rf_grid.best_score_)

print("\n===== LightGBM =====")
print("Best Parameters:", lgbm_grid.best_params_)
print("Best CV F1:", lgbm_grid.best_score_)


===== SVM =====
Best Parameters: {'model__C': 10, 'model__kernel': 'linear', 'tfidf__max_df': 0.95, 'tfidf__min_df': 2, 'tfidf__ngram_range': (1, 2), 'tfidf__sublinear_tf': True}
Best CV F1: 0.9444601894183874

===== Random Forest =====
Best Parameters: {'model__max_depth': None, 'model__min_samples_split': 5, 'model__n_estimators': 100, 'tfidf__max_df': 0.95, 'tfidf__min_df': 3, 'tfidf__ngram_range': (1, 1), 'tfidf__sublinear_tf': False}
Best CV F1: 0.8909059146989051

===== LightGBM =====
Best Parameters: {'model__learning_rate': 0.1, 'model__n_estimators': 100, 'model__num_leaves': 31, 'tfidf__max_df': 0.95, 'tfidf__min_df': 3, 'tfidf__ngram_range': (1, 2), 'tfidf__sublinear_tf': False}
Best CV F1: 0.9099022489291893


In [ ]:
print("\n===== SVM =====")
print("Best Parameters:", svm_grid.best_params_)
print("Best CV F1:", svm_grid.best_score_)

In [55]:
model = SVC(C=10,kernel='linear')
tfidf =TfidfVectorizer(max_df=0.95,min_df=2,ngram_range=(1,2),sublinear_tf=True)
X_trainF =tfidf.fit_transform(X_train)
X_testF =tfidf.transform(X_test)
model.fit(X_trainF,y_train)
pre=model.predict(X_testF)
score=model.decision_function(X_testF)


print(f"Accuracy: {accuracy_score(y_test, pre):.4f}")
print(f"Precision: {precision_score(y_test, pre):.4f}")
print(f"Recall: {recall_score(y_test, pre):.4f}")
print(f"F1-Score: {f1_score(y_test, pre):.4f}")
print(f"Confusion Matrix:\n{confusion_matrix(y_test, pre)}")
print(f"ROC-AUC: {roc_auc_score(y_test, score):.4f}")
print(f"PR-AUC: {average_precision_score(y_test, score):.4f}")

Accuracy: 0.9883
Precision: 0.9857
Recall: 0.9262
F1-Score: 0.9550
Confusion Matrix:
[[964   2]
 [ 11 138]]
ROC-AUC: 0.9880
PR-AUC: 0.9785


In [72]:
text="Hi, Your Profile has Selected at Tech Mahindra".lower().strip()

In [73]:
import re

In [74]:
text=re.sub(r'[^\w\s]',"",text)

In [75]:
text=tfidf.transform([text])

In [76]:
a=model.predict(text)

In [77]:
a

array([0])